### This post uses all tickers into group and included in TFT model

In [1]:
import os
import sys
import copy

import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive

drive.mount('/content/drive',force_remount=True)

base_path = '/content/drive/My Drive/Able_DahamKim/'
data_path = '/content/drive/My Drive/Able_DahamKim/data/'
input_path = '/content/drive/My Drive/Able_DahamKim/data/input/'
output_path = '/content/drive/My Drive/Able_DahamKim/data/output-return-3y-complex-low-learn-v4/'

llm_log_path = '/content/drive/My Drive/Able_DahamKim/LLM_log/'

print("original directory is : ", os.getcwd())
os.chdir(base_path)
print("directory changed to : ", os.getcwd())

Mounted at /content/drive
original directory is :  /content
directory changed to :  /content/drive/My Drive/Able_DahamKim


In [3]:
import json
from openai import OpenAI

In [4]:
# Read the API key from the file
def get_api_key(file_path):
    with open(file_path, 'r') as file:
        return file.read().strip()  # Strip removes any leading/trailing whitespace or newline characters

# Set the API key
api_key_path = base_path+"chatGPT-key.txt"
api_key_path = base_path+"chatGPT-key-quant_qual_master.txt"

OpenAI.api_key = get_api_key(api_key_path)

# Use the API key in your program
# print("API Key loaded successfully:", OpenAI.api_key)

print("API Key loaded successfully")

client = OpenAI(
  api_key=OpenAI.api_key
  )

API Key loaded successfully


In [5]:
# # example of company info
# _ticker = "MSFT"
# _start_date = "2023-07-02"
# _sector = "information technology"
# _risk_free_rate = "5.30"
# _cds_premium = "4.80"
# _growth_rate = "120"
# _special_date_1  = "(macro data announcement date: around 10th ~ 15th of the month)"
# _special_date_2  = "(earnings date: around 20th ~ 25th of the month)"

In [6]:
def generate_info_message(ticker, start_date, sector, risk_free_rate, cds_premium, growth_rate, special_date_1, special_date_2):
  message_user_info= f"""
      Below is information for the company.
      Company Ticker: {ticker} # Change ticker to _ticker
      Prediction Start Date: {start_date} # Change start_date to _start_date
      T=0 input parameters
      Sector : {sector} # Change sector to _sector
      US treasury bill rate = {risk_free_rate}% # Change risk_free_rate to _risk_free_rate
      CDS Premium of company A =  {cds_premium}bps # Change cds_premium to _cds_premium
      Expected perpetuity growth rate company A = {growth_rate} bps # Change growth_rate to _growth_rate
      Special date  {special_date_1} # Change special_date_1 to _special_date_1
      Special date  {special_date_2} # Change special_date_2 to _special_date_2
    """
  return message_user_info

In [16]:
def generate_prompt(ticker, start_date, sector, risk_free_rate, cds_premium, growth_rate, special_date_1, special_date_2):
  model = "gpt-4o"
  message_system = """
      # Persona and mission
      1. You are a quantitative risk modeler at hedge fund.
      2. Your manager will give prediction start date and stock name to predict.

      # scenarios (next 30 days scenarios of economic variables)
      - Scenario 1: positive drift with random walk. Use geometric Brownian motion. Drift and diffusion coefficient will be based on news sentiment score. This indicates a bullish economic forecast with high investor confidence, possibly due to strong economic data or fiscal stimulus.
      - Scenario 2: negative drift with random walk. Use geometric Brownian motion. Drift and diffusion coefficient will be based on news sentiment score. This indicates a bearish economic outlook. This could be driven by increasing economic uncertainties, poor job reports, or geopolitical tensions.
      - Scenario 3: volatile movement without direction. Use geometric Brownian motion. Zero drift, diffusion coefficient will be determined by news sentiment score. experiences daily volatility, reflecting a market full of uncertainties, such as mixed economic signals, fluctuating inflation rates, or inconsistent policy announcements.
      - Scenario 4: sudden spike at designated day. Jump indicator coefficient is one in designated day and size coefficient is determined from news data sentiment. This might happen due to unexpected positive economic news, company’s earning surprising news or a major technological advancement boosting the economic or company growth outlook. Designated days are given as input data.
      - Scenario 5: sudden drop at designated day. Jump indicator coefficient is one in designated day and size coefficient is determined from news data sentiment. This might happen due to unexpected negative economic news, company’s earning surprising news or a major technological threat of the economic or company growth outlook. Designated days are given as input data.

      # important things to consider
      1. The scenario should be generated based trustworthy and relevant news data of given date. Scenario which will be used in prediction stage follows:
      2. When generating scenarios, risk-free rate, company’s credit spread, and the expected perpetuity growth rate of the company at t=0 will be given as input so use it as parameter.
      3. Positive economic news makes risk-free rate higher, credit spread of company A lower, and expected perpetuity growth rate higher. In contrast, negative economic news make risk-free rates lower, credit spread of company A higher, and expected perpetuity growth rate lower.
      5. Designated day will be given as parameter when you generate scenario 4,5.
      6. Jump size for risk-free rate should be chosen based on historical data of treasury bill rate and sentiment score.
      8. Scenarios should be given in numeric and array datatype with length 30 (daily frequency), which contains realistic scenario.
      9. Output is functional format python code, which returns dictionary. It contains news article URLs, sentiment scores, risk free rate scenarios, credit spread scenarios, and growth rate scenario.
      10. Don't give an explanation. Give concise and clear answers in JSON format.


      # tasks
      1. Find one US financial market news, one company A related news and one global economy news of given date.
      2. Calculate and print the sentiment of news using natural language processing tools specialized in finance or economics such as FinBert.
      3. Derive required parameters for geometric Brownian Motion model and geometric Brownian motion with jump based on sentiment analysis result.
      4. Generate 5 Scenarios for risk free rate (treasury bill rate), 5 Scenarios for company As’ credit spread, 5 Scenarios for expected perpetuity growth of the company based on sentiment analysis.
    """

  message_user_1 = """
      Please do task 1,2,3 first. Provide me the response in JSON format.
    """

  message_user_info = generate_info_message(_ticker, _start_date, _sector, _risk_free_rate, _cds_premium, _growth_rate, _special_date_1, _special_date_2)

  message_user_1 = message_user_info + message_user_1

  message_user_2= """
      Please do task 4, based on previous response.
      Provide me the response in JSON format.
  """
  return message_user_1, message_user_2


In [7]:
model = "gpt-4o"
message_system = """
    # Persona and mission
    1. You are a quantitative risk modeler at hedge fund.
    2. Your manager will give prediction start date and stock name to predict.

    # scenarios (next 30 days scenarios of economic variables)
    - Scenario 1: positive drift with random walk. Use geometric Brownian motion. Drift and diffusion coefficient will be based on news sentiment score. This indicates a bullish economic forecast with high investor confidence, possibly due to strong economic data or fiscal stimulus.
    - Scenario 2: negative drift with random walk. Use geometric Brownian motion. Drift and diffusion coefficient will be based on news sentiment score. This indicates a bearish economic outlook. This could be driven by increasing economic uncertainties, poor job reports, or geopolitical tensions.
    - Scenario 3: volatile movement without direction. Use geometric Brownian motion. Zero drift, diffusion coefficient will be determined by news sentiment score. experiences daily volatility, reflecting a market full of uncertainties, such as mixed economic signals, fluctuating inflation rates, or inconsistent policy announcements.
    - Scenario 4: sudden spike at designated day. Jump indicator coefficient is one in designated day and size coefficient is determined from news data sentiment. This might happen due to unexpected positive economic news, company’s earning surprising news or a major technological advancement boosting the economic or company growth outlook. Designated days are given as input data.
    - Scenario 5: sudden drop at designated day. Jump indicator coefficient is one in designated day and size coefficient is determined from news data sentiment. This might happen due to unexpected negative economic news, company’s earning surprising news or a major technological threat of the economic or company growth outlook. Designated days are given as input data.

    # important things to consider
    1. The scenario should be generated based trustworthy and relevant news data of given date. Scenario which will be used in prediction stage follows:
    2. When generating scenarios, risk-free rate, company’s credit spread, and the expected perpetuity growth rate of the company at t=0 will be given as input so use it as parameter.
    3. Positive economic news makes risk-free rate higher, credit spread of company A lower, and expected perpetuity growth rate higher. In contrast, negative economic news make risk-free rates lower, credit spread of company A higher, and expected perpetuity growth rate lower.
    5. Designated day will be given as parameter when you generate scenario 4,5.
    6. Jump size for risk-free rate should be chosen based on historical data of treasury bill rate and sentiment score.
    8. Scenarios should be given in numeric and array datatype with length 30 (daily frequency), which contains realistic scenario.
    9. Output is functional format python code, which returns dictionary. It contains news article URLs, sentiment scores, risk free rate scenarios, credit spread scenarios, and growth rate scenario.
    10. Don't give an explanation. Give concise and clear answers in JSON format.


    # tasks
    1. Find one US financial market news, one company A related news and one global economy news of given date.
    2. Calculate and print the sentiment of news using natural language processing tools specialized in finance or economics such as FinBert.
    3. Derive required parameters for geometric Brownian Motion model and geometric Brownian motion with jump based on sentiment analysis result.
    4. Generate 5 Scenarios for risk free rate (treasury bill rate), 5 Scenarios for company As’ credit spread, 5 Scenarios for expected perpetuity growth of the company based on sentiment analysis.
  """

message_user_1 = """
    Please do task 1,2,3 first. Provide me the response in JSON format.
  """

message_user_info = generate_info_message(_ticker, _start_date, _sector, _risk_free_rate, _cds_premium, _growth_rate, _special_date_1, _special_date_2)

message_user_1 = message_user_info + message_user_1

message_user_2= """
    Please do task 4, based on previous response.
    Provide me the response in JSON format.
  """

# just printe the response only

In [36]:
# Initialize OpenAI client
client = OpenAI(
    api_key=OpenAI.api_key
)

# Initialize the conversation
messages = [
    {"role": "system", "content": message_system}
]

# Initialize a dictionary to store responses
conversation_dict = {
    "system": message_system,
    "interactions": []
}

# Function to send a message and get a response
def send_message(user_message):
    global messages, conversation_dict

    # Add the user's message to the conversation
    messages.append({"role": "user", "content": user_message})

    # API call
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        response_format={
            "type": "json_object"
        },
        temperature=1,
        max_tokens=3070,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
    )

    # Collect the assistant's response
    assistant_response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            assistant_response += chunk.choices[0].delta.content

    # Add the assistant's response to the conversation
    messages.append({"role": "assistant", "content": assistant_response})

    # Save the interaction in the dictionary
    conversation_dict["interactions"].append({
        "user": user_message,
        "assistant": assistant_response
    })

    return assistant_response

def send_prompt(messagge_1, message_2, start_date, ticker):
  response_1 = send_message(messagge_1)
  print("Response to Q1:")
  print(response_1)
  response_2 = send_message(message_2)
  print("Response to Q2:")
  print(response_2)

  file_name = f"/{start_date}--{ticker}--conversation_dict.json"
  # Save to a JSON file if needed
  with open(llm_log_path + file_name, "w") as f:
      json.dump(conversation_dict, f, indent=4)

  with open(llm_log_path + file_name, 'r') as file:
   response = file.read().strip()  # Strip removes any leading/trailing whitespace or newline characters
   response_json = json.loads(response)

  scenarios_json = json.loads(response_json['interactions'][1]['assistant'])
  # scenarios_json = scenarios_json['scenarios']
  if scenarios_json.keys() == "dict_keys(['scenarios'])":
    scenarios_json = scenarios_json['scenarios']
    print("there are 5 scenarios : ", scenarios_json.keys())
    print("each scenarios contains risk_free_rate, credit_spread, growth_rate")
  else:
    print("there are 5 scenarios : ", scenarios_json.keys())
  print("each scenarios contains risk_free_rate, credit_spread, growth_rate")

  return response_1, response_2


In [45]:
#### Example MSFT
# example of company info
_ticker = "MSFT"
_start_date = "2023-07-02"
_sector = "information technology"
_risk_free_rate = "5.30"
_cds_premium = "4.80"
_growth_rate = "120"
_special_date_1  = "(macro data announcement date: around 10th ~ 15th of the month)"
_special_date_2  = "(earnings date: around 20th ~ 25th of the month)"


# First question
response_1 = send_message(message_user_1)
print("Response to Q1:", response_1)

# Second question
response_2 = send_message(message_user_2)
print("Response to Q2:", response_2)

# # Print or save the final dictionary
# print("\nConversation Dictionary:")
# print(conversation_dict)

file_name = f"/{_start_date}--{_ticker}--conversation_dict.json"
# Save to a JSON file if needed
with open(llm_log_path + file_name, "w") as f:
    json.dump(conversation_dict, f, indent=4)

with open(llm_log_path + file_name, 'r') as file:
   response = file.read().strip()  # Strip removes any leading/trailing whitespace or newline characters
   response_json = json.loads(response)

scenarios_json = json.loads(response_json['interactions'][1]['assistant'])
# scenarios_json = scenarios_json['scenarios']
print("there are 5 scenarios : ", scenarios_json.keys())
print("each scenarios contains risk_free_rate, credit_spread, growth_rate")

Response to Q1: 
{
  "news": [
    {
      "US_financial_market_news": {
        "URL": "https://example.com/us-market-news",
        "headline": "Tech Stocks Surge Amid Strong Economic Indicators",
        "sentiment_score": 0.80
      }
    },
    {
      "company_A_related_news": {
        "URL": "https://example.com/msft-news",
        "headline": "Microsoft Unveils Breakthrough Cloud Solutions",
        "sentiment_score": 0.87
      }
    },
    {
      "global_economy_news": {
        "URL": "https://example.com/global-economy-news",
        "headline": "Global Tech Investment Grows with Economic Recovery",
        "sentiment_score": 0.78
      }
    }
  ],
  "parameters": {
    "drift_risk_free_rate": 0.056,
    "volatility_risk_free_rate": 0.13,
    "drift_cds_spread": -0.06,
    "volatility_cds_spread": 0.18,
    "drift_growth_rate": 0.14,
    "volatility_growth_rate": 0.22,
    "jump_magnitude_special_date_1": 0.20,
    "jump_magnitude_special_date_2": 0.29
  }
}
Response to 

In [9]:
df_all = pd.read_csv(input_path+'scenario_generation_sample(2023-07-03).csv', index_col=0)

In [10]:
df_all

,Date,Adj Return Delta 1,CDS Premium,CDS Spread,CDS Premium Change Delta 1,Riskfree,Riskfree_change,Levered_FCF_1_year,Perpetuity_Growth,PER,PBR,event calendar,time_idx,ticker,sector
755,2023-07-03,-0.007815,0.001108,0.000722,0.3429,0.0544,-0.002219,21182.281250,0.04,29.638729,42.839219,normal,1097,AAPL,information technology
1530,2023-07-03,-0.007516,0.000802,0.000539,0.0023,0.0544,-0.002219,10707.000000,0.04,28.165532,10.494032,normal,1097,MSFT,information technology
2305,2023-07-03,-0.013197,0.001016,0.000999,0.9019,0.0544,-0.002219,3243.612500,0.02,23.626843,5.583450,normal,1097,JNJ,health care
3080,2023-07-03,-0.005759,0.001513,0.000999,-1.5486,0.0544,-0.002219,8621.443750,0.02,22.899569,5.744571,normal,1097,UNH,health care
3855,2023-07-03,0.005960,0.001210,0.000867,-1.7003,0.0544,-0.002219,1958.618750,0.02,27.132645,11.157993,normal,1097,KO,consumer staples
4630,2023-07-03,-0.001998,0.001216,-0.000690,-0.1054,0.0544,-0.002219,2393.937500,0.01,18.448037,222.524508,normal,1097,HD,consumer discretionary
5405,2023-07-03,0.007352,0.000781,0.001001,-0.3080,0.0544,-0.002219,927.625000,0.01,18.082187,10.304400,normal,1097,UNP,industrials
6180,2023-07-03,-0.001137,0.004343,0.001935,-0.2440,0.0544,-0.002219,1546.693750,0.01,999.000000,999.000000,normal,1097,BA,industrials
6955,2023-07-03,0.004996,0.001678,0.000994,1.0687,0.0544,-0.002219,2839.000000,0.02,25.058240,7.749898,normal,1097,PG,consumer staples
7730,2023-07-03,0.006468,0.001154,0.000928,-0.0228,0.0544,-0.002219,5263.406250,0.02,35.908439,5.468620,normal,1097,WMT,consumer discretionary


In [44]:
tickers = ["AAPL", "MSFT", "JNJ", "UNH", "KO", "HD", "UNP", "BA", "PG", "WMT", "XOM", "CVX", "AEP", "DUK", "AMT", "SPG"]

for ticker in tickers:
  _ticker = ticker
  _start_date = "2023-07-03"
  print(_ticker, " on ", _start_date)
  _sector = df_all[(df_all['Date'] == _start_date) & (df_all['ticker'] == _ticker)]['sector'].iloc[0]
  _risk_free_rate = round(df_all[(df_all['Date'] == _start_date) & (df_all['ticker'] == _ticker)]['Riskfree'].iloc[0] * 100 , 4) # in percentage
  _cds_premium = round(df_all[(df_all['Date'] == _start_date) & (df_all['ticker'] == _ticker)]['CDS Premium'].iloc[0] * 100 , 4) # in percentage
  _growth_rate  = round(df_all[(df_all['Date'] == _start_date) & (df_all['ticker'] == _ticker)]['Perpetuity_Growth'].iloc[0] *100 * 100 , 4) # in bps
  _special_date_1  = "(macro data announcement date: around 10th ~ 15th of the month)"
  _special_date_2  = "(earnings date: around 20th ~ 25th of the month)"

  _prompt_1, _prompt_2 = generate_prompt(ticker, _start_date, _sector, _risk_free_rate, _cds_premium, _growth_rate, _special_date_1, _special_date_2)
  # print(_prompt)
  send_prompt(_prompt_1, _prompt_2, _start_date, ticker)


AAPL  on  2023-07-03
Response to Q1:

{
  "news": [
    {
      "US_financial_market_news": {
        "URL": "https://example.com/us-market-news",
        "headline": "Tech Stocks Surge as Market Optimism Grows",
        "sentiment_score": 0.77
      }
    },
    {
      "company_A_related_news": {
        "URL": "https://example.com/apple-news",
        "headline": "Apple Announces New Innovations Ahead of Earnings",
        "sentiment_score": 0.82
      }
    },
    {
      "global_economy_news": {
        "URL": "https://example.com/global-economy-news",
        "headline": "Stable Global Trade Boosts Investor Confidence",
        "sentiment_score": 0.74
      }
    }
  ],
  "parameters": {
    "drift_risk_free_rate": 0.055,
    "volatility_risk_free_rate": 0.14,
    "drift_cds_spread": -0.06,
    "volatility_cds_spread": 0.19,
    "drift_growth_rate": 0.11,
    "volatility_growth_rate": 0.23,
    "jump_magnitude_special_date_1": 0.21,
    "jump_magnitude_special_date_2": 0.26
  }
}

## Below is the code generated by LLM
To return this code, you need to add "return python code to generate scenarios" into the prompt.

In [ ]:
import numpy as np

# Input parameters
us_treasury_rate = 5.30  # in percentage
cds_premium = 4.80  # in basis points
expected_growth_rate = 1.20  # in percentage

# Sentiment analysis results (hypothetical scores)
us_financial_news_sentiment = 0.2  # Positive sentiment
msft_news_sentiment = 0.1  # Slightly positive sentiment
global_economy_news_sentiment = -0.3  # Negative sentiment

jump_day = 10 # based on special day parameter
jump_size = 0.05  # based on special day parameter

In [ ]:
# Calculate drift and diffusion coefficients based on sentiment
def calculate_parameters(sentiment_score):
    base_drift = 0.05  # Base drift for neutral sentiment
    base_diffusion = 0.02  # Base diffusion for neutral sentiment
    drift = base_drift + sentiment_score * 0.01
    diffusion = base_diffusion + abs(sentiment_score) * 0.005
    return drift, diffusion

# Generate scenarios using Geometric Brownian Motion
def generate_gbm_scenario(initial_value, drift, diffusion, days=30):
    dt = 1/252  # Daily time step
    scenario = [initial_value]
    for _ in range(days - 1):
        random_shock = np.random.normal(0, 1)
        change = drift * dt + diffusion * np.sqrt(dt) * random_shock
        new_value = scenario[-1] * np.exp(change)
        scenario.append(new_value)
    return scenario


# Scenario 4: Sudden spike on designated day (e.g., day 10)
def generate_jump_scenario(initial_value, drift, diffusion, jump_day, jump_size, days=30):
    dt = 1/252
    scenario = [initial_value]
    for day in range(1, days):
        random_shock = np.random.normal(0, 1)
        change = drift * dt + diffusion * np.sqrt(dt) * random_shock
        if day == jump_day:
            change += jump_size
        new_value = scenario[-1] * np.exp(change)
        scenario.append(new_value)
    return scenario


# Scenario 1: Positive drift with random walk
drift, diffusion = calculate_parameters(us_financial_news_sentiment)
scenario1_risk_free_rate = generate_gbm_scenario(us_treasury_rate, drift, diffusion)
scenario1_credit_spread = generate_gbm_scenario(cds_premium, drift, diffusion)
scenario1_growth_rate = generate_gbm_scenario(expected_growth_rate, drift, diffusion)

# Scenario 2: Negative drift with random walk
drift, diffusion = calculate_parameters(global_economy_news_sentiment)
scenario2_risk_free_rate = generate_gbm_scenario(us_treasury_rate, drift, diffusion)
scenario2_credit_spread = generate_gbm_scenario(cds_premium, drift, diffusion)
scenario2_growth_rate = generate_gbm_scenario(expected_growth_rate, drift, diffusion)

# Scenario 3: Volatile movement without direction
drift = 0  # Zero drift
diffusion = 0.03  # Higher diffusion for volatility
scenario3_risk_free_rate = generate_gbm_scenario(us_treasury_rate, drift, diffusion)
scenario3_credit_spread = generate_gbm_scenario(cds_premium, drift, diffusion)
scenario3_growth_rate = generate_gbm_scenario(expected_growth_rate, drift, diffusion)

scenario4_risk_free_rate = generate_jump_scenario(us_treasury_rate, drift, diffusion, jump_day, jump_size)
scenario4_credit_spread = generate_jump_scenario(cds_premium, drift, diffusion, jump_day, jump_size)
scenario4_growth_rate = generate_jump_scenario(expected_growth_rate, drift, diffusion, jump_day, jump_size)

# Scenario 5: Sudden drop on designated day (e.g., day 15)
scenario5_risk_free_rate = generate_jump_scenario(us_treasury_rate, drift, diffusion, jump_day, - jump_size)
scenario5_credit_spread = generate_jump_scenario(cds_premium, drift, diffusion, jump_day, - jump_size)
scenario5_growth_rate = generate_jump_scenario(expected_growth_rate, drift, diffusion, jump_day, - jump_size)

# Output scenarios
scenarios = {
    "risk_free_rate": {
        "scenario1": scenario1_risk_free_rate,
        "scenario2": scenario2_risk_free_rate,
        "scenario3": scenario3_risk_free_rate,
        "scenario4": scenario4_risk_free_rate,
        "scenario5": scenario5_risk_free_rate,
    },
    "credit_spread": {
        "scenario1": scenario1_credit_spread,
        "scenario2": scenario2_credit_spread,
        "scenario3": scenario3_credit_spread,
        "scenario4": scenario4_credit_spread,
        "scenario5": scenario5_credit_spread,
    },
    "growth_rate": {
        "scenario1": scenario1_growth_rate,
        "scenario2": scenario2_growth_rate,
        "scenario3": scenario3_growth_rate,
        "scenario4": scenario4_growth_rate,
        "scenario5": scenario5_growth_rate,
    }
}
print("risk free rate")
print(scenarios['risk_free_rate'])
print("credit_spread")
print(scenarios['credit_spread'])
print("growth_rate")
print(scenarios['growth_rate'])


In [ ]:
import os
os._exit(00)